# Week 3 – Sentiment Analysis & Urgency Scoring

**Project:** AI-Driven Citizen Grievance & Sentiment Analysis System  
**Goal:** Classify the **emotional tone** of each complaint as `Positive`, `Neutral`, `Negative`, or `Critical/Urgent`, and assign a **numeric urgency score (0–10)** so authorities can triage issues faster.

**Approach (two tracks):**

| Track | Model | When to use |
|-------|-------|-------------|
| A | Rule-based baseline (VADER + keyword rules) | Fast, no GPU needed |
| B | Fine-tuned DistilBERT Transformer | Higher accuracy, GPU optional |

**Prerequisite:** Week 1 must have produced `output/cleaned_mapping.csv`  
with at least a `clean_text` column.  Week 2 model artifacts are optionally used.

## 1 – Install & Import Dependencies

In [ ]:
# Install required packages (run once)
# Uncomment lines below if running for the first time
!pip install transformers datasets torch accelerate vaderSentiment scikit-learn matplotlib seaborn joblib -q

In [ ]:
import os, sys

# 1. Find out what file Python tried to import as 'nltk'
# (it may have failed, so check sys.modules carefully)
if 'nltk' in sys.modules:
    print("nltk loaded from:", sys.modules['nltk'].__file__)
else:
    print("nltk not yet imported — checking working directory...")

# 2. List any shadowing files in current directory
cwd = os.getcwd()
print(f"\nWorking directory: {cwd}")
suspects = [f for f in os.listdir(cwd) if f.lower().startswith('nltk')]
print("Conflicting files found:", suspects if suspects else "None (check subfolders or __pycache__)")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings, re, joblib, json
warnings.filterwarnings("ignore")

# NLTK
import nltk
nltk.download('vader_lexicon', quiet=True)
nltk.download('punkt', quiet=True)
from nltk.sentiment.vader import SentimentIntensityAnalyzer

# Sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, accuracy_score, f1_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline

# HuggingFace Transformers
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline as hf_pipeline
)
from datasets import Dataset, DatasetDict

out = Path("output")
out.mkdir(exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
print("All imports OK")

## 2 – Load and Inspect Data from Week 1

In [ ]:
df = pd.read_excel("CategoryCode_Mapping.xlsx")
print(f"Loaded: {df.shape}")
print("Columns:", df.columns.tolist())
df.head(3)

In [ ]:
# ── Choose the text column ───────────────────────────────────────────────────
# Week 1 produces 'clean_text' via NLP preprocessing.
# We also keep 'description' (or similar) as raw text for display.

TEXT_COL = "clean_text"          # preprocessed text (Week 1 output)
RAW_COL  = "description"         # raw text column – update if different

# Fallback: if clean_text is absent, derive it from description
if TEXT_COL not in df.columns:
    candidate_cols = [c for c in df.columns if 'desc' in c.lower() or 'complaint' in c.lower() or 'text' in c.lower()]
    print(f"'{TEXT_COL}' not found. Candidate columns: {candidate_cols}")
    if candidate_cols:
        TEXT_COL = candidate_cols[0]
        print(f"Using '{TEXT_COL}' as text column.")

# Drop rows with missing text
data = df[[TEXT_COL]].copy()
if RAW_COL in df.columns:
    data[RAW_COL] = df[RAW_COL]
if "source_sheet" in df.columns:
    data["department"] = df["orgcode"]

data = data.dropna(subset=[TEXT_COL])
data = data[data[TEXT_COL].str.strip() != ""]
data = data.reset_index(drop=True)

print(f"Working dataset size: {len(data)} rows")
data.head(3)


## 3 – Track A: Rule-Based Baseline with Urgency Keywords

VADER (Valence Aware Dictionary and sEntiment Reasoner) is a lexicon-based tool
tuned for social-media / short texts. We extend it with a custom urgency keyword
list to catch civic emergency language ("danger", "flood", "dead", etc.).

In [ ]:
import os, sys

# Remove the broken partial import so Python will retry cleanly
if 'nltk' in sys.modules:
    del sys.modules['nltk']

# Find the conflicting file
cwd = os.getcwd()
print("Working directory:", cwd)

for root, dirs, files in os.walk(cwd):
    for f in files:
        if f.lower() in ('nltk.py', 'nltk.pyc'):
            print("FOUND CONFLICT:", os.path.join(root, f))

# Also check sys.path entries
print("\nsys.path:")
for p in sys.path:
    if p:
        try:
            for f in os.listdir(p):
                if f.lower() in ('nltk.py', 'nltk'):
                    print(f"  CONFLICT in {p}: {f}")
        except:
            pass

In [ ]:
import subprocess
subprocess.run(["pip", "install", "--force-reinstall", "nltk"], check=True)

In [ ]:
import subprocess
subprocess.run(["pip", "uninstall", "-y", "nltk"], check=True)
subprocess.run(["pip", "install", "nltk"], check=True)

In [ ]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer
# ── VADER Analyser ────────────────────────────────────────────────────────────
vader = SentimentIntensityAnalyzer()

# ── Urgency keyword list (domain-specific for civic grievances) ───────────────
URGENCY_KEYWORDS = {
    "critical": [
        "emergency", "urgent", "immediately", "critical", "dangerous",
        "hazard", "life-threatening", "death", "dead", "collapse",
        "flood", "fire", "explosion", "gas leak", "sewage overflow",
        "accident", "injured", "hospitalized", "children at risk",
        "contaminated water", "power outage", "blackout", "no water",
        "road blocked", "bridge collapse"
    ],
    "negative": [
        "broken", "damaged", "not working", "failure", "problem",
        "complaint", "disgusting", "terrible", "horrible", "worst",
        "months", "weeks", "neglect", "ignored", "no response",
        "poor", "pathetic", "unacceptable", "slow", "delay"
    ],
    "positive": [
        "resolved", "fixed", "excellent", "satisfied", "thank",
        "great work", "appreciate", "improvement", "good service"
    ]
}

def count_keywords(text, keyword_list):
    """Count how many keywords from a list appear in the text."""
    text_lower = str(text).lower()
    return sum(1 for kw in keyword_list if kw in text_lower)

print("VADER + Urgency keyword rules ready.")

In [ ]:
def vader_sentiment_label(text):
    """
    Returns (sentiment_label, urgency_score).

    Sentiment labels:
      - Critical/Urgent : contains urgency keywords OR compound <= -0.6
      - Negative        : compound < -0.05
      - Neutral         : -0.05 <= compound <= 0.05
      - Positive        : compound > 0.05

    Urgency score (0-10):
      Combines VADER negativity, keyword matches and sentence intensity.
    """
    scores = vader.polarity_scores(str(text))
    compound = scores['compound']
    neg      = scores['neg']

    # Keyword counts
    critical_hits  = count_keywords(text, URGENCY_KEYWORDS["critical"])
    negative_hits  = count_keywords(text, URGENCY_KEYWORDS["negative"])
    positive_hits  = count_keywords(text, URGENCY_KEYWORDS["positive"])

    # ── Sentiment label ──────────────────────────────────────────────────────
    if critical_hits > 0 or compound <= -0.60:
        label = "Critical/Urgent"
    elif compound < -0.05 or negative_hits > 0:
        label = "Negative"
    elif compound > 0.05 or positive_hits > 0:
        label = "Positive"
    else:
        label = "Neutral"

    # ── Urgency score (0–10) ─────────────────────────────────────────────────
    # Base: map compound from [-1, 1] to [0, 10] inverted (more negative = higher urgency)
    base_score  = (1 - compound) / 2 * 7           # 0–7 from sentiment
    kw_boost    = min(critical_hits * 1.5 + negative_hits * 0.5, 3.0)  # 0–3 boost
    urgency_score = round(min(base_score + kw_boost, 10.0), 2)

    return label, urgency_score, compound, critical_hits, negative_hits


# Apply to dataset
results = data[TEXT_COL].apply(lambda x: vader_sentiment_label(x))
data["sentiment_label"]  = results.apply(lambda x: x[0])
data["urgency_score"]    = results.apply(lambda x: x[1])
data["vader_compound"]   = results.apply(lambda x: x[2])
data["critical_kw_hits"] = results.apply(lambda x: x[3])
data["negative_kw_hits"] = results.apply(lambda x: x[4])

print("Sentiment + Urgency scoring complete.")
print("\nLabel distribution:")
print(data["sentiment_label"].value_counts())
print(f"\nAverage urgency score: {data['urgency_score'].mean():.2f}")

In [ ]:
sns.set_theme(style="whitegrid")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── Plot 1: Sentiment label distribution ────────────────────────────────────
label_order = ["Positive", "Neutral", "Negative", "Critical/Urgent"]
label_colors = ["#2ecc71", "#95a5a6", "#e67e22", "#e74c3c"]
vc = data["sentiment_label"].value_counts().reindex(label_order).fillna(0)
axes[0].bar(vc.index, vc.values, color=label_colors, edgecolor="black")
axes[0].set_title("Sentiment Label Distribution", fontsize=13)
axes[0].set_xlabel("Sentiment")
axes[0].set_ylabel("Count")
for i, v in enumerate(vc.values):
    axes[0].text(i, v + 5, str(int(v)), ha='center', fontsize=9)

# ── Plot 2: Urgency score distribution ──────────────────────────────────────
axes[1].hist(data["urgency_score"], bins=30, color="#3498db", edgecolor="black")
axes[1].set_title("Urgency Score Distribution (0–10)", fontsize=13)
axes[1].set_xlabel("Urgency Score")
axes[1].set_ylabel("Frequency")

# ── Plot 3: Urgency score by sentiment label (box plot) ──────────────────────
plot_data = data[data["sentiment_label"].isin(label_order)]
sns.boxplot(
    data=plot_data, x="sentiment_label", y="urgency_score",
    order=label_order, palette=label_colors, ax=axes[2]
)
axes[2].set_title("Urgency Score by Sentiment Label", fontsize=13)
axes[2].set_xlabel("Sentiment")
axes[2].set_ylabel("Urgency Score")

plt.suptitle("Track A – VADER Baseline: Sentiment & Urgency Overview", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(out / "week3_vader_overview.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved to output/week3_vader_overview.png")

## 4 – Label Creation Strategy

Since the dataset has **no ground-truth sentiment labels**, we use two strategies:

1. **Silver labels**: Use the VADER-based rule above to auto-label the entire corpus.
2. **Manual annotation (recommended)**: Label ~500 samples by hand and use them as
   a held-out test set to evaluate Transformer fine-tuning.

Here we proceed with **silver labels** to demonstrate the full pipeline.

In [ ]:
# ── Encode silver sentiment labels for model training ────────────────────────
LABEL_MAP = {
    "Positive":       0,
    "Neutral":        1,
    "Negative":       2,
    "Critical/Urgent": 3
}
ID2LABEL = {v: k for k, v in LABEL_MAP.items()}

data["label_id"] = data["sentiment_label"].map(LABEL_MAP)

# Drop any unmapped rows (shouldn't happen, but safety check)
data = data.dropna(subset=["label_id"]).copy()
data["label_id"] = data["label_id"].astype(int)

print("Label mapping:", LABEL_MAP)
print("Label distribution:")
print(data["label_id"].value_counts().sort_index().rename(index=ID2LABEL))

In [ ]:
# ── Train / Validation / Test split ─────────────────────────────────────────
train_df, temp_df = train_test_split(
    data[[TEXT_COL, "label_id", "urgency_score", "sentiment_label"]],
    test_size=0.30,
    stratify=data["label_id"],
    random_state=42
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label_id"],
    random_state=42
)

print(f"Train : {len(train_df):,} samples")
print(f"Val   : {len(val_df):,} samples")
print(f"Test  : {len(test_df):,} samples")

## 5 – Track A Continued: TF-IDF + Logistic Regression (Sentiment Classifier)

A strong, fast baseline before we move to Transformers.

In [ ]:
# ── TF-IDF + Logistic Regression pipeline ────────────────────────────────────
tfidf_lr_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=15_000,
        ngram_range=(1, 2),
        sublinear_tf=True,
        min_df=2
    )),
    ("clf", LogisticRegression(
        max_iter=1000, C=1.0, random_state=42,
        solver="lbfgs", 
    ))
])

# ── Cross-validation on training set ─────────────────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = cross_validate(
    tfidf_lr_pipeline,
    train_df[TEXT_COL],
    train_df["label_id"],
    cv=cv,
    scoring=["accuracy", "f1_macro", "f1_weighted"],
    return_train_score=False,
    n_jobs=-1
)

print("── TF-IDF + LR Cross-Validation (5-fold) ──")
print(f"  Accuracy   : {cv_results['test_accuracy'].mean():.4f} ± {cv_results['test_accuracy'].std():.4f}")
print(f"  F1-Macro   : {cv_results['test_f1_macro'].mean():.4f} ± {cv_results['test_f1_macro'].std():.4f}")
print(f"  F1-Weighted: {cv_results['test_f1_weighted'].mean():.4f} ± {cv_results['test_f1_weighted'].std():.4f}")

# ── Fit on full train set and evaluate on test set ────────────────────────────
tfidf_lr_pipeline.fit(train_df[TEXT_COL], train_df["label_id"])
y_pred_lr = tfidf_lr_pipeline.predict(test_df[TEXT_COL])

print("\n── Classification Report (Test Set) ──")
print(classification_report(
    test_df["label_id"], y_pred_lr,
    target_names=[ID2LABEL[i] for i in sorted(ID2LABEL)]
))


In [ ]:
# ── Confusion matrix for TF-IDF + LR ─────────────────────────────────────────
cm = confusion_matrix(test_df["label_id"], y_pred_lr)
fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[ID2LABEL[i] for i in sorted(ID2LABEL)]
)
disp.plot(ax=ax, colorbar=True, cmap="Blues", xticks_rotation=30)
ax.set_title("Confusion Matrix – TF-IDF + Logistic Regression (Sentiment)")
plt.tight_layout()
plt.savefig(out / "week3_tfidf_lr_cm.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: output/week3_tfidf_lr_cm.png")

## 6 – Track B: Fine-Tuning DistilBERT for Sentiment Classification

DistilBERT is a lightweight, 40% smaller version of BERT that retains ~97% of  
BERT's performance. We fine-tune it on our silver-labelled complaint data.

> **Note:** Fine-tuning takes ~5–15 minutes on GPU / ~30–60 min on CPU.  
> Set `USE_TRANSFORMER = False` below to skip training and jump to inference.

In [ ]:
# ── Toggle: set False to skip fine-tuning (uses TF-IDF+LR instead) ───────────
USE_TRANSFORMER = False      # ← change to False on CPU-only or time-constrained runs
MODEL_NAME = "distilbert-base-uncased"
NUM_LABELS = len(LABEL_MAP)
MAX_LEN    = 128             # max token length (complaints are short)
BATCH_SIZE = 16
EPOCHS     = 3

print(f"Transformer fine-tuning: {'ENABLED' if USE_TRANSFORMER else 'SKIPPED'}")
print(f"Model: {MODEL_NAME}, Labels: {NUM_LABELS}, Device: {DEVICE}")


In [ ]:
if USE_TRANSFORMER:
    # ── Load tokenizer ────────────────────────────────────────────────────────
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    def tokenize_function(examples):
        return tokenizer(
            examples["text"],
            padding="max_length",
            truncation=True,
            max_length=MAX_LEN
        )

    # ── Build HuggingFace DatasetDict ─────────────────────────────────────────
    def make_hf_dataset(df_split):
        return Dataset.from_dict({
            "text":   df_split[TEXT_COL].tolist(),
            "labels": df_split["label_id"].tolist()
        })

    raw_ds = DatasetDict({
        "train": make_hf_dataset(train_df),
        "val":   make_hf_dataset(val_df),
        "test":  make_hf_dataset(test_df)
    })

    tokenized_ds = raw_ds.map(tokenize_function, batched=True)

    print("Tokenization complete.")
    print(tokenized_ds)

In [ ]:
if USE_TRANSFORMER:
    # ── Load pre-trained model with classification head ───────────────────────
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_LABELS,
        id2label=ID2LABEL,
        label2id=LABEL_MAP,
        ignore_mismatched_sizes=True
    )
    model = model.to(DEVICE)
    print(f"Model loaded: {MODEL_NAME} → {NUM_LABELS} classes")
    print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    """Compute accuracy and macro-F1 during Trainer evaluation."""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1_macro": f1_score(labels, predictions, average="macro"),
        "f1_weighted": f1_score(labels, predictions, average="weighted")
    }

if USE_TRANSFORMER:
    training_args = TrainingArguments(
        output_dir=str(out / "distilbert_sentiment"),
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=EPOCHS,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        logging_steps=50,
        fp16=(DEVICE == "cuda"),     # mixed precision on GPU
        report_to="none",            # disable W&B / MLflow
        seed=42
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_ds["train"],
        eval_dataset=tokenized_ds["val"],
        compute_metrics=compute_metrics
    )

    print("Trainer configured. Starting fine-tuning...")

In [ ]:
if USE_TRANSFORMER:
    train_output = trainer.train()
    print("\n── Training Summary ──")
    print(f"  Training loss : {train_output.training_loss:.4f}")
    print(f"  Train runtime : {train_output.metrics.get('train_runtime', 0):.1f}s")

In [ ]:
if USE_TRANSFORMER:
    # ── Evaluate on held-out test set ─────────────────────────────────────────
    test_results = trainer.evaluate(tokenized_ds["test"])
    print("── DistilBERT Test Set Evaluation ──")
    for k, v in test_results.items():
        if isinstance(v, float):
            print(f"  {k}: {v:.4f}")

    # Detailed classification report
    test_preds = trainer.predict(tokenized_ds["test"])
    y_pred_bert = np.argmax(test_preds.predictions, axis=-1)
    y_true_bert = test_preds.label_ids

    print("\n── Classification Report (DistilBERT – Test Set) ──")
    print(classification_report(
        y_true_bert, y_pred_bert,
        target_names=[ID2LABEL[i] for i in sorted(ID2LABEL)]
    ))

In [ ]:
if USE_TRANSFORMER:
    cm_bert = confusion_matrix(y_true_bert, y_pred_bert)
    fig, ax = plt.subplots(figsize=(8, 6))
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm_bert,
        display_labels=[ID2LABEL[i] for i in sorted(ID2LABEL)]
    )
    disp.plot(ax=ax, colorbar=True, cmap="Purples", xticks_rotation=30)
    ax.set_title("Confusion Matrix – DistilBERT Fine-Tuned (Sentiment)")
    plt.tight_layout()
    plt.savefig(out / "week3_distilbert_cm.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: output/week3_distilbert_cm.png")

In [ ]:
if USE_TRANSFORMER:
    # ── Save fine-tuned model and tokenizer ───────────────────────────────────
    bert_save_path = out / "distilbert_sentiment_final"
    trainer.save_model(str(bert_save_path))
    tokenizer.save_pretrained(str(bert_save_path))
    print(f"Model saved to: {bert_save_path}")

## 7 – Urgency Score Refinement

We upgrade the simple VADER-based urgency score by incorporating:
- Model-predicted sentiment confidence (probability of Critical/Urgent)
- Keyword density
- Text length (longer complaints may indicate more severe issues)
- Complaint age/repetition (placeholder — integrate with Week 2 department labels)

In [ ]:
def compute_urgency_score_v2(
    text: str,
    sentiment_label: str,
    vader_compound: float,
    critical_kw_hits: int,
    negative_kw_hits: int,
    sentiment_proba: float = None   # optional: P(Critical/Urgent) from ML model
) -> float:
    """
    Composite urgency score (0–10).

    Components:
      1. Sentiment weight  (40%): based on label category
      2. VADER intensity   (20%): uses compound score inverted
      3. Keyword boost     (25%): critical & negative keyword density
      4. Text length       (10%): proxy for detail level
      5. Model confidence  (5%):  P(Critical/Urgent) from fine-tuned model
    """
    # 1. Sentiment weight
    SENT_WEIGHT = {
        "Critical/Urgent": 4.0,
        "Negative":        2.5,
        "Neutral":         1.0,
        "Positive":        0.0
    }
    sentiment_component = SENT_WEIGHT.get(sentiment_label, 1.0)

    # 2. VADER intensity (map compound [-1,1] → [0, 2])
    vader_component = (1 - vader_compound) / 1.0   # 0 = very positive, 2 = very negative

    # 3. Keyword density boost (capped)
    word_count = max(len(str(text).split()), 1)
    kw_density = (critical_kw_hits * 2.0 + negative_kw_hits * 0.5) / word_count * 50
    kw_component = min(kw_density, 2.5)

    # 4. Text length component (longer = more detail = potentially more urgent)
    length_component = min(np.log1p(word_count) / np.log1p(50), 1.0)

    # 5. Model confidence (if available)
    model_component = float(sentiment_proba) if sentiment_proba is not None else 0.0

    # Weighted sum → scale to 0–10
    raw_score = (
        0.40 * sentiment_component +
        0.20 * vader_component +
        0.25 * kw_component +
        0.10 * length_component +
        0.05 * model_component * 10
    )

    # Normalize to 0–10
    max_possible = 0.40 * 4.0 + 0.20 * 2.0 + 0.25 * 2.5 + 0.10 * 1.0 + 0.05 * 10.0
    urgency_score = round(min(raw_score / max_possible * 10, 10.0), 2)
    return urgency_score


# Apply refined urgency scoring
data["urgency_score_v2"] = data.apply(
    lambda row: compute_urgency_score_v2(
        text=row[TEXT_COL],
        sentiment_label=row["sentiment_label"],
        vader_compound=row["vader_compound"],
        critical_kw_hits=row["critical_kw_hits"],
        negative_kw_hits=row["negative_kw_hits"]
    ),
    axis=1
)

print("Refined urgency scores computed.")
print(data[["sentiment_label", "urgency_score", "urgency_score_v2"]].describe().round(3))

## 8 – Model Comparison: Track A vs Track B

In [ ]:
# Build a summary comparison table
comparison = []

# Track A1: VADER rule-based
vader_acc = accuracy_score(test_df["label_id"], test_df["sentiment_label"].map(LABEL_MAP))
vader_f1  = f1_score(test_df["label_id"], test_df["sentiment_label"].map(LABEL_MAP), average="macro", zero_division=0)
comparison.append({"Model": "VADER Rule-Based (baseline)", "Accuracy": vader_acc, "F1-Macro": vader_f1})

# Track A2: TF-IDF + LR
lr_acc = accuracy_score(test_df["label_id"], y_pred_lr)
lr_f1  = f1_score(test_df["label_id"], y_pred_lr, average="macro", zero_division=0)
comparison.append({"Model": "TF-IDF + Logistic Regression", "Accuracy": lr_acc, "F1-Macro": lr_f1})

# Track B: DistilBERT (only if trained)
if USE_TRANSFORMER:
    bert_acc = accuracy_score(y_true_bert, y_pred_bert)
    bert_f1  = f1_score(y_true_bert, y_pred_bert, average="macro", zero_division=0)
    comparison.append({"Model": "DistilBERT Fine-Tuned", "Accuracy": bert_acc, "F1-Macro": bert_f1})

comp_df = pd.DataFrame(comparison).sort_values("F1-Macro", ascending=False)
comp_df.to_csv(out / "week3_model_comparison.csv", index=False)
comp_df.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
palette = sns.color_palette("Set2", len(comp_df))

for ax, metric in zip(axes, ["Accuracy", "F1-Macro"]):
    bars = ax.barh(comp_df["Model"], comp_df[metric], color=palette, edgecolor="black")
    ax.set_xlabel(metric)
    ax.set_title(f"Sentiment Classifier – {metric}")
    ax.set_xlim(0, 1)
    for bar, val in zip(bars, comp_df[metric]):
        ax.text(val + 0.005, bar.get_y() + bar.get_height()/2, f"{val:.3f}", va='center')

plt.suptitle("Week 3 – Sentiment Model Comparison", fontsize=14)
plt.tight_layout()
plt.savefig(out / "week3_model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: output/week3_model_comparison.png")

## 9 – Department-wise Urgency Analysis

Combining the Week 2 department classification with Week 3 urgency scores,  
we can answer: **Which departments receive the most critical complaints?**

In [ ]:
if "department" in data.columns:
    dept_urgency = (
        data.groupby("department")["urgency_score_v2"]
        .agg(["mean", "max", "count"])
        .rename(columns={"mean": "avg_urgency", "max": "peak_urgency", "count": "total_complaints"})
        .sort_values("avg_urgency", ascending=False)
        .reset_index()
    )
    dept_urgency.to_csv(out / "week3_dept_urgency.csv", index=False)
    print("Department urgency summary:")
    display(dept_urgency.round(3))

    # Stacked bar: sentiment distribution per department
    dept_sentiment = (
        data.groupby(["department", "sentiment_label"])
        .size().unstack(fill_value=0)
    )
    # Reorder columns
    col_order = [c for c in ["Positive", "Neutral", "Negative", "Critical/Urgent"] if c in dept_sentiment.columns]
    dept_sentiment = dept_sentiment[col_order]

    dept_sentiment_pct = dept_sentiment.div(dept_sentiment.sum(axis=1), axis=0) * 100

    fig, ax = plt.subplots(figsize=(14, 6))
    dept_sentiment_pct.plot(kind="bar", stacked=True, ax=ax,
                            color=["#2ecc71", "#95a5a6", "#e67e22", "#e74c3c"],
                            edgecolor="white")
    ax.set_title("Sentiment Distribution by Department (%)", fontsize=13)
    ax.set_xlabel("Department")
    ax.set_ylabel("Percentage of Complaints")
    ax.legend(title="Sentiment", bbox_to_anchor=(1.01, 1), loc="upper left")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(out / "week3_dept_sentiment_dist.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: output/week3_dept_sentiment_dist.png")
else:
    print("'department' column not found – skipping department-wise analysis.")
    print("Ensure Week 2 is run and 'source_sheet' is present in cleaned_mapping.csv.")

## 10 – Priority Dashboard: Top Critical Complaints

In [ ]:
# ── Top Critical/Urgent complaints sorted by urgency score ───────────────────
critical_complaints = (
    data[data["sentiment_label"] == "Critical/Urgent"]
    .sort_values("urgency_score_v2", ascending=False)
    .head(20)
    .reset_index(drop=True)
)

display_cols = [TEXT_COL, "urgency_score_v2", "vader_compound", "critical_kw_hits"]
if "department" in data.columns:
    display_cols.insert(1, "department")

print(f"Total Critical/Urgent complaints: {len(data[data['sentiment_label'] == 'Critical/Urgent']):,}")
print("\n── Top 20 Most Urgent Complaints ──")
display(critical_complaints[display_cols].head(20))

# Save full scored dataset
final_cols = [TEXT_COL, "sentiment_label", "urgency_score_v2",
              "urgency_score", "vader_compound", "critical_kw_hits", "negative_kw_hits"]
if "department" in data.columns:
    final_cols.insert(1, "department")

data[final_cols].to_csv(out / "week3_sentiment_scored.csv", index=False)
print("\nFull scored dataset saved to: output/week3_sentiment_scored.csv")

In [ ]:
# ── Urgency score heatmap by sentiment and department ────────────────────────
if "department" in data.columns and data["department"].nunique() <= 20:
    pivot = data.pivot_table(
        values="urgency_score_v2",
        index="department",
        columns="sentiment_label",
        aggfunc="mean"
    ).fillna(0)

    fig, ax = plt.subplots(figsize=(12, 7))
    sns.heatmap(
        pivot, annot=True, fmt=".1f",
        cmap="YlOrRd", linewidths=0.5, ax=ax,
        cbar_kws={"label": "Avg Urgency Score"}
    )
    ax.set_title("Average Urgency Score by Department × Sentiment Label", fontsize=13)
    ax.set_xlabel("Sentiment Label")
    ax.set_ylabel("Department")
    plt.xticks(rotation=30, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(out / "week3_urgency_heatmap.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: output/week3_urgency_heatmap.png")
else:
    print("Skipping heatmap (department column absent or too many unique values).")

## 11 – Save All Artifacts

In [ ]:
# ── Save TF-IDF + LR sentiment pipeline ──────────────────────────────────────
joblib.dump(tfidf_lr_pipeline, out / "sentiment_tfidf_lr_model.pkl")

# ── Save label mappings ───────────────────────────────────────────────────────
with open(out / "week3_label_map.json", "w") as f:
    json.dump({"label_map": LABEL_MAP, "id2label": {str(k): v for k, v in ID2LABEL.items()}}, f, indent=2)

# ── Save urgency keyword list ─────────────────────────────────────────────────
with open(out / "week3_urgency_keywords.json", "w") as f:
    json.dump(URGENCY_KEYWORDS, f, indent=2)

print("── Saved Artifacts ──")
for p in sorted(out.glob("week3_*")):
    print(f"  {p}")

## 12 – End-to-End Inference Demo

Full pipeline: raw complaint text → sentiment label + urgency score.

In [ ]:
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text: str) -> str:
    """Identical to Week 1 preprocessing pipeline."""
    if not text:
        return ""
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(t) for t in tokens
              if t not in stop_words and len(t) > 2]
    return " ".join(tokens)


def analyze_complaint(raw_text: str, use_bert: bool = False) -> dict:
    """
    Full pipeline: raw complaint → sentiment label + urgency score.

    Parameters
    ----------
    raw_text : str   Original citizen complaint
    use_bert : bool  Use DistilBERT model if available, else TF-IDF+LR

    Returns
    -------
    dict with sentiment_label, urgency_score, vader_compound, confidence
    """
    clean = preprocess_text(raw_text)

    # VADER scores for urgency components
    vader_scores = vader.polarity_scores(raw_text)
    compound     = vader_scores['compound']
    crit_hits    = count_keywords(raw_text, URGENCY_KEYWORDS['critical'])
    neg_hits     = count_keywords(raw_text, URGENCY_KEYWORDS['negative'])

    # Sentiment prediction
    if use_bert and USE_TRANSFORMER:
        # DistilBERT inference
        bert_pipe = hf_pipeline(
            "text-classification",
            model=str(out / "distilbert_sentiment_final"),
            tokenizer=str(out / "distilbert_sentiment_final"),
            device=0 if DEVICE == "cuda" else -1
        )
        result     = bert_pipe(raw_text[:512], truncation=True)[0]
        label      = result['label']
        confidence = result['score']
    else:
        # TF-IDF + LR inference
        proba      = tfidf_lr_pipeline.predict_proba([clean])[0]
        pred_id    = np.argmax(proba)
        label      = ID2LABEL[pred_id]
        confidence = proba[pred_id]

    # Urgency score
    urgency = compute_urgency_score_v2(
        text=raw_text,
        sentiment_label=label,
        vader_compound=compound,
        critical_kw_hits=crit_hits,
        negative_kw_hits=neg_hits
    )

    return {
        "complaint": raw_text,
        "sentiment_label": label,
        "urgency_score": urgency,
        "confidence": round(confidence, 4),
        "vader_compound": round(compound, 4),
        "critical_kw_hits": crit_hits
    }


# ── Test complaints ───────────────────────────────────────────────────────────
test_complaints = [
    "The water supply has been contaminated – children are getting sick. This is a health emergency!",
    "Garbage not collected from our area for over 3 weeks. Very unhygienic situation.",
    "The new park in our locality is very nice. Thank you for the improvement.",
    "Street lights near the school are off since last month. Please fix.",
    "A large pothole on Main Street caused an accident today. Urgent repair needed!",
    "The sewage line is completely blocked and overflowing onto the road – dangerous!",
]

results_list = [analyze_complaint(c) for c in test_complaints]
results_demo = pd.DataFrame(results_list)

print("── End-to-End Inference Results ──")
pd.set_option('display.max_colwidth', 60)
display(results_demo.sort_values("urgency_score", ascending=False).reset_index(drop=True))

## 13 – Week 3 Summary

| Step | What was done |
|------|---------------|
| VADER Baseline | Rule-based sentiment labels + urgency score v1 |
| TF-IDF + LR | Supervised sentiment classifier (Track A) |
| DistilBERT | Fine-tuned Transformer classifier (Track B) |
| Urgency Score v2 | Composite scoring: sentiment + VADER + keywords + length |
| Department Analysis | Urgency heatmap + sentiment distribution per department |
| Priority Queue | Top Critical/Urgent complaints ranked by urgency score |
| Outputs | CSVs, model artifacts, plots, keyword JSON |

**Output files:**
- `output/week3_sentiment_scored.csv` — full scored dataset
- `output/week3_tfidf_lr_sentiment.pkl` — TF-IDF+LR pipeline
- `output/distilbert_sentiment_final/` — fine-tuned DistilBERT (if trained)
- `output/week3_model_comparison.csv` — model accuracy comparison
- `output/week3_urgency_keywords.json` — custom urgency keyword list

**Next steps (Week 4):** Hyperparameter tuning (Optuna), SHAP/LIME explainability,  
integration with Week 2 department classifier into a unified REST API endpoint.